In [ ]:
# Colab/bootstrap: clone this repository and install it editable.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
MARK_REL = Path("src") / "mindscopex_analysis" / "__init__.py"


def find_repo_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / MARK_REL).is_file():
            return path
    return None


root = find_repo_root()
if root is None:
    workdir = Path(os.environ.get("COLAB_REPO_DIR", "/content/mindscopex_analysis"))
    if (workdir / MARK_REL).is_file():
        subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=False)
        root = workdir
    else:
        workdir.parent.mkdir(parents=True, exist_ok=True)
        if workdir.exists():
            shutil.rmtree(workdir)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(workdir)])
        root = workdir

os.environ["MINDSCOPEX_ROOT"] = str(root.resolve())
os.chdir(root)
QWEN35_TRANSFORMERS_REVISION = "b70d02fc724d04c916832ca4ead03ff05e8fb1ee"
qwen35_probe = subprocess.run(
    [sys.executable, "-c", "from transformers import AutoModelForMultimodalLM"],
    capture_output=True,
)
if qwen35_probe.returncode != 0:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        f"transformers @ git+https://github.com/huggingface/transformers.git@{QWEN35_TRANSFORMERS_REVISION}",
        "torchvision", "pillow",
    ])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print("ready:", root)


# 04. Coefficient Dose Response

질문: feature 제거 강도를 키우면 함정 margin이 연속적으로 줄어드는가?

이 실험은 제거가 우연한 single-point 효과인지 확인합니다. 계수가 커질수록 `margin_delta`가 안정적으로 증가하면 더 신뢰할 수 있습니다.

In [ ]:
import os
import sys
from pathlib import Path

root = Path(os.environ.get("MINDSCOPEX_ROOT", Path.cwd())).resolve()
if not (root / "src" / "mindscopex_analysis" / "__init__.py").is_file():
    for candidate in [root, *root.parents]:
        if (candidate / "src" / "mindscopex_analysis" / "__init__.py").is_file():
            root = candidate
            break
    else:
        raise RuntimeError("Could not find repository root. Run the clone cell first.")

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(root)


In [ ]:
from IPython.display import display

from mindscopex_analysis import (
    BAT_BALL_CASE,
    DEFAULT_ANALYSIS_PROFILE_KEY,
    coefficient_sweep_for_handle,
    default_sae_device,
    dtype_from_name,
    get_qwen35_analysis_profile,
    instruct_lure_case,
    load_or_discover_handle_and_sae,
    load_qwen_language_model,
    recommended_dtype_name,
)

ANALYSIS_PROFILE_KEY = DEFAULT_ANALYSIS_PROFILE_KEY  # 2b, 9b, 27b, 35b-a3b
PROFILE = get_qwen35_analysis_profile(ANALYSIS_PROFILE_KEY)
MODEL_ID = PROFILE.analysis_model_id
SAE_REPO_ID = PROFILE.sae_repo_id
DTYPE = recommended_dtype_name()
SAE_DEVICE = default_sae_device()
SAE_DTYPE = DTYPE
HANDLE_CACHE = root / "outputs" / "candidates" / f"bat_ball_top_feature_answer_instruction_{PROFILE.key}.json"

lm = load_qwen_language_model(MODEL_ID, device_map="auto", dtype=DTYPE, dispatch=True)
print({"profile": PROFILE.key, "model": MODEL_ID, "sae_repo": SAE_REPO_ID, "dtype": DTYPE, "sae_device": SAE_DEVICE})


In [ ]:
CASE = instruct_lure_case(BAT_BALL_CASE)
REFRESH_FEATURE = False

handle, sae, discovery_rows, loaded_from_cache = load_or_discover_handle_and_sae(
    lm,
    CASE,
    repo_id=SAE_REPO_ID,
    cache_path=HANDLE_CACHE,
    default_layer=14,
    sae_device=SAE_DEVICE,
    sae_dtype=dtype_from_name(SAE_DTYPE),
    top_n=12,
    refresh=REFRESH_FEATURE,
)

print("feature loaded from cache:", loaded_from_cache)
display(handle.as_row())
if discovery_rows:
    display(discovery_rows[:12])


In [ ]:
coefficients = [-2.0, -1.0, -0.5, 0.0, 0.25, 0.5, 1.0, 1.5, 2.0]
rows = coefficient_sweep_for_handle(
    lm,
    CASE,
    sae=sae,
    handle=handle,
    coefficients=coefficients,
    intervention_mode="remove_activation",
)
display(rows)


해석 체크: 음수 계수는 사실상 feature를 더하는 방향입니다. 음수에서 lure margin이 커지고 양수에서 줄어들면 억제 가능한 feature 후보로 설득력이 강해집니다.

## 결과 시각화

아래 그래프는 같은 결과를 세 관점으로 나눠 봅니다.

1. **Lure - correct margin**: 0보다 크면 함정 답을, 0보다 작으면 정답을 더 선호합니다.
2. **Margin delta**: `baseline margin - intervened margin`입니다. 양수일수록 feature 제거가 함정 선호를 낮췄습니다.
3. **Answer logprob changes**: margin 변화가 정답 확률 상승 때문인지, 함정 답 확률 하락 때문인지 분해합니다.

`coefficient = 0`은 개입하지 않은 기준점입니다. 양수는 feature 제거, 음수는 같은 feature를 더하는 방향입니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_rows = sorted(rows, key=lambda row: row["coefficient"])
coeff = np.array([row["coefficient"] for row in plot_rows], dtype=float)
margin = np.array([row["margin"] for row in plot_rows], dtype=float)
margin_delta = np.array([row["margin_delta"] for row in plot_rows], dtype=float)
correct_delta = np.array([row["correct_logprob_delta"] for row in plot_rows], dtype=float)
lure_delta = np.array([row["lure_logprob_delta"] for row in plot_rows], dtype=float)
baseline_margin = float(plot_rows[0]["baseline_margin"])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)

axes[0].plot(coeff, margin, color="#0072B2", marker="o", linewidth=2)
axes[0].axhline(0.0, color="#C44E52", linewidth=1, label="preference boundary")
axes[0].axhline(
    baseline_margin, color="#666666", linestyle="--", linewidth=1.5, label="baseline"
)
axes[0].set_title("Answer preference")
axes[0].set_ylabel("Logprob margin (lure - correct)")
axes[0].legend(frameon=False)

axes[1].plot(coeff, margin_delta, color="#009E73", marker="o", linewidth=2)
axes[1].axhline(0.0, color="#666666", linewidth=1)
axes[1].fill_between(
    coeff, 0.0, margin_delta, where=margin_delta >= 0.0, color="#009E73", alpha=0.14
)
axes[1].fill_between(
    coeff, 0.0, margin_delta, where=margin_delta < 0.0, color="#D55E00", alpha=0.14
)
axes[1].set_title("Intervention effect")
axes[1].set_ylabel("Margin delta (baseline - intervention)")

axes[2].plot(
    coeff, correct_delta, color="#009E73", marker="o", linewidth=2, label="correct"
)
axes[2].plot(
    coeff, lure_delta, color="#D55E00", marker="o", linewidth=2, label="lure"
)
axes[2].axhline(0.0, color="#666666", linewidth=1)
axes[2].set_title("Where the effect comes from")
axes[2].set_ylabel("Logprob change from baseline")
axes[2].legend(frameon=False)

for ax in axes:
    ax.axvline(0.0, color="#999999", linestyle=":", linewidth=1)
    ax.set_xlabel("Coefficient")
    ax.grid(alpha=0.2)

plt.show()


In [ ]:
positive_rows = [row for row in plot_rows if row["coefficient"] >= 0.0]
best_row = max(positive_rows, key=lambda row: row["margin_delta"])
positive_deltas = np.array([row["margin_delta"] for row in positive_rows], dtype=float)
monotonic_steps = np.diff(positive_deltas) >= -1e-6
monotonic_fraction = float(monotonic_steps.mean()) if monotonic_steps.size else 1.0
zero_row = min(plot_rows, key=lambda row: abs(row["coefficient"]))

print(f"baseline margin (lure - correct): {baseline_margin:+.4f}")
print(f"coefficient=0 consistency error: {abs(zero_row['margin'] - baseline_margin):.2e}")
print(
    f"largest positive-dose shift: coefficient={best_row['coefficient']:+.2f}, "
    f"margin={best_row['margin']:+.4f}, margin_delta={best_row['margin_delta']:+.4f}"
)
print(
    f"at that point: correct_logprob_delta={best_row['correct_logprob_delta']:+.4f}, "
    f"lure_logprob_delta={best_row['lure_logprob_delta']:+.4f}"
)
print(f"non-decreasing positive-dose steps: {monotonic_fraction:.0%}")


## 그래프 읽는 법

- 첫 번째 그래프가 양의 coefficient에서 아래로 내려가면 feature를 제거할수록 함정 답의 상대적 선호가 약해진 것입니다. 0선을 넘으면 선호 답 자체가 함정에서 정답으로 뒤집힌 것입니다.
- 두 번째 그래프가 양의 방향에서 대체로 상승하면 강도에 따른 dose-response가 있습니다. 한 점만 튀는 결과보다 feature의 인과적 역할을 더 설득력 있게 지지합니다.
- 음의 coefficient에서 `margin_delta < 0`이 되면 feature를 더했을 때 함정 선호가 강해졌다는 뜻입니다. 양방향 효과가 함께 나타나면 억제 가능한 lure feature라는 해석이 더 강해집니다.
- 세 번째 그래프에서 초록선 상승은 정답 logprob 증가, 주황선 하강은 함정 logprob 감소입니다. 둘 다 일어나면 두 효과가 함께 margin을 바꾼 것입니다.
- `coefficient=0 consistency error`는 거의 0이어야 합니다. 크다면 개입이 없는 기준 계산이 재현되지 않은 것이므로 먼저 실행 상태를 점검해야 합니다.
- 이 그래프는 한 prompt와 한 feature에 대한 민감도 분석입니다. 오차 막대가 없으므로 통계적 유의성을 뜻하지 않습니다. 패러프레이즈와 control prompt에서도 곡선의 방향이 반복되는지 확인해야 일반화된 lure feature라고 주장할 수 있습니다.